# LeetCode 1678 — Goal Parser Interpretation bằng Python

Link đề: https://leetcode.com/problems/goal-parser-interpretation/

Độ khó: **Easy** · Chủ đề: String

Skeleton LeetCode:

```python
class Solution:
    def interpret(self, command: str) -> str:

```

Mục tiêu học:

1. Hiểu bài toán **phân tích cú pháp** (parsing) đơn giản: đọc chuỗi, nhận ra các "token" `G`, `()`, `(al)`.
2. Dùng `str.replace` đúng thứ tự và hiểu **vì sao thứ tự quan trọng**.
3. Viết bộ quét (scanner) một lượt với **nhìn trước** (lookahead) một ký tự: chỉ cần xem tối đa 2 ký tự để quyết định token.
4. Biết xây chuỗi kết quả hiệu quả bằng `list` + `"".join` thay vì cộng chuỗi lặp.
5. Làm quen thêm cách bảng tra `dict` theo token và biểu thức chính quy `re.sub`.
6. Tự viết lại và giải các biến thể "giải mã chuỗi theo bảng ký hiệu".

## 1. Bài toán yêu cầu gì?

Chuỗi `command` chỉ gồm ba loại "mảnh" (token) nối nhau theo thứ tự bất kỳ:

| Token | Dịch thành |
|---|---|
| `"G"` | `"G"` |
| `"()"` | `"o"` |
| `"(al)"` | `"al"` |

Dịch từng token rồi nối theo đúng thứ tự gốc. Trả về chuỗi kết quả.

| command | Kết quả |
|---|---|
| `"G()(al)"` | `"Goal"` |
| `"G()()()()(al)"` | `"Gooooal"` |
| `"(al)G(al)()()G"` | `"alGalooG"` |

Constraints:

- `1 <= command.length <= 100`
- `command` chỉ gồm `"G"`, `"()"`, `"(al)"` theo thứ tự nào đó — tức là input **luôn hợp lệ**.

### Tách token bằng tay

Ví dụ 3: `"(al)G(al)()()G"`

```text
vị trí:  0123 4 5678 9A BC D
token :  (al) G (al) () () G
dịch  :  al   G al   o  o  G   -> "alGalooG"
```

Quan sát quan trọng (cũng là hint của đề): khi gặp `(`, chỉ cần nhìn **ký tự kế tiếp**:

- kế tiếp là `)` → token `()` dài 2 → `"o"`.
- kế tiếp là `a` → token `(al)` dài 4 → `"al"`.

In [1]:
command = "(al)G(al)()()G"
for i, ch in enumerate(command):
    print(i, repr(ch))

0 '('
1 'a'
2 'l'
3 ')'
4 'G'
5 '('
6 'a'
7 'l'
8 ')'
9 '('
10 ')'
11 '('
12 ')'
13 'G'


## 2. Nền tảng Python cần biết

### 2.1. Chuỗi bất biến và `str.replace`

`s.replace(old, new)` trả về **chuỗi mới**, thay **mọi** lần xuất hiện của `old` bằng `new`. Chuỗi gốc không đổi (chuỗi là immutable).

In [2]:
s = "G()(al)"
t = s.replace("()", "o")
print("s =", s)
print("t =", t)
print(t.replace("(al)", "al"))

s = G()(al)
t = Go(al)
Goal


### 2.2. Gọi nối tiếp (method chaining)

Vì `replace` trả về chuỗi, có thể gọi tiếp: `s.replace(a, b).replace(c, d)` — thực hiện từ trái sang phải.

In [3]:
print("G()(al)".replace("()", "o").replace("(al)", "al"))

Goal


### 2.3. Truy cập ký tự và nhìn trước `s[i + 1]`

`s[i]` là ký tự thứ `i`. Khi đọc `s[i + 1]`, phải chắc `i + 1 < len(s)`. Ở bài này input hợp lệ nên sau `(` luôn còn ký tự, nhưng thói quen kiểm tra biên vẫn quan trọng.

In [4]:
s = "G()"
i = 1
print("s[i] =", s[i], "| s[i+1] =", s[i + 1])
try:
    print(s[3])
except IndexError as e:
    print("IndexError:", e)

s[i] = ( | s[i+1] = )
IndexError: string index out of range


### 2.4. Xây chuỗi bằng `list` + `"".join`

Cộng chuỗi `res += "o"` nhiều lần tạo chuỗi mới mỗi lần. Cách chuẩn: gom các mảnh vào `list` bằng `append`, cuối cùng `"".join(parts)` một lần.

In [5]:
parts = []
for piece in ["G", "o", "al"]:
    parts.append(piece)
    print("parts =", parts)
print("".join(parts))

parts = ['G']
parts = ['G', 'o']
parts = ['G', 'o', 'al']
Goal


### 2.5. `startswith(prefix, i)` — kiểm tra token tại vị trí i

`s.startswith("(al)", i)` trả `True` nếu tại vị trí `i` bắt đầu đúng chuỗi `"(al)"` — không cần cắt slice.

In [6]:
s = "G()(al)"
print(s.startswith("()", 1), s.startswith("(al)", 3), s.startswith("(al)", 1))

True True False


## 3. Cách trực giác — hai lần `replace`

Ý tưởng đầu tiên của nhiều người: thay `"()"` bằng `"o"`, rồi thay `"(al)"` bằng `"al"`. Chữ `G` giữ nguyên.

In [7]:
def interpret_replace(command: str) -> str:
    return command.replace("()", "o").replace("(al)", "al")

print(interpret_replace("G()(al)"))
print(interpret_replace("G()()()()(al)"))
print(interpret_replace("(al)G(al)()()G"))

Goal
Gooooal
alGalooG


### Có vấn đề gì không?

Lời giải này **đúng** và hoàn toàn nộp được. Nhưng cần hiểu **vì sao** đúng, vì cách `replace` dễ sai khi bài phức tạp hơn:

- Ta phải chắc rằng thay `"()"` không phá vỡ `"(al)"`. Ở đây `"(al)"` không chứa `"()"`, nên an toàn.
- Mỗi `replace` duyệt **toàn bộ** chuỗi một lần → nhiều token = nhiều lượt duyệt; mỗi lần tạo một chuỗi mới.
- Nếu bảng token có **chồng lấn** (một token là phần của token khác, hoặc kết quả thay thế tạo ra token mới), thứ tự `replace` có thể cho kết quả sai. Xem lỗi 2 ở mục 12.

Cách bền vững hơn: **quét một lượt từ trái sang phải**, nhận diện token tại chỗ.

In [8]:
# Thứ tự ngược lại vẫn đúng ở bài này, vì hai token không chồng lấn:
print("G()(al)".replace("(al)", "al").replace("()", "o"))

Goal


## 4. Insight — nhìn trước 1 ký tự là đủ

Tại vị trí `i`:

| `command[i]` | `command[i+1]` | Token | Dịch | Bước nhảy |
|---|---|---|---|---:|
| `G` | (không cần xem) | `G` | `G` | 1 |
| `(` | `)` | `()` | `o` | 2 |
| `(` | `a` | `(al)` | `al` | 4 |

Vì input luôn hợp lệ, không có trường hợp nào khác. Đây là một **máy quét** (lexer) tí hon: đọc ký tự đầu, nhìn trước tối đa một ký tự, quyết định token, nhảy qua token đó.

In [9]:
command = "G()(al)"
i = 1
print("command[i] =", command[i], "| nhìn trước:", command[i + 1], "-> token '()' -> 'o', nhảy 2")
i = 3
print("command[i] =", command[i], "| nhìn trước:", command[i + 1], "-> token '(al)' -> 'al', nhảy 4")

command[i] = ( | nhìn trước: ) -> token '()' -> 'o', nhảy 2
command[i] = ( | nhìn trước: a -> token '(al)' -> 'al', nhảy 4


## 5. Lời giải chính — quét một lượt với lookahead

Giảng từng đoạn:

**Đoạn 1 — chuẩn bị:** `parts = []`, `i = 0`.

**Đoạn 2 — vòng `while i < len(command)`:** dùng `while` vì bước nhảy thay đổi (1, 2 hoặc 4).

**Đoạn 3 — ba nhánh:** `G` → thêm `"G"`, `i += 1`; `(` + `)` → thêm `"o"`, `i += 2`; còn lại → thêm `"al"`, `i += 4`.

**Đoạn 4 — trả `"".join(parts)`.**

In [10]:
def interpret_scan(command: str) -> str:
    parts = []
    i = 0
    while i < len(command):
        if command[i] == "G":
            parts.append("G")
            i += 1
        elif command[i + 1] == ")":
            parts.append("o")
            i += 2
        else:
            parts.append("al")
            i += 4
    return "".join(parts)

print(interpret_scan("G()(al)"))
print(interpret_scan("G()()()()(al)"))
print(interpret_scan("(al)G(al)()()G"))

Goal
Gooooal
alGalooG


Nhánh `elif command[i + 1] == ")"` chỉ chạy khi `command[i] != "G"`, tức `command[i] == "("`. Với input hợp lệ, sau `(` luôn còn ít nhất một ký tự, nên `command[i + 1]` không bao giờ vượt biên.

## 6. TRACE từng bước

Trace `command = "(al)G(al)()()G"`:

| i | command[i] | nhìn trước | token | thêm | i mới | kết quả tạm |
|---:|:---:|:---:|---|---|---:|---|
| 0 | ( | a | (al) | al | 4 | al |
| 4 | G | — | G | G | 5 | alG |
| 5 | ( | a | (al) | al | 9 | alGal |
| 9 | ( | ) | () | o | 11 | alGalo |
| 11 | ( | ) | () | o | 13 | alGaloo |
| 13 | G | — | G | G | 14 | alGalooG |

`i = 14 = len(command)` → dừng.

In [11]:
def trace_interpret(command: str) -> str:
    parts = []
    i = 0
    print(f"command = {command!r}, len = {len(command)}")
    while i < len(command):
        ch = command[i]
        if ch == "G":
            token, out, step, peek = "G", "G", 1, "—"
        elif command[i + 1] == ")":
            token, out, step, peek = "()", "o", 2, command[i + 1]
        else:
            token, out, step, peek = "(al)", "al", 4, command[i + 1]
        parts.append(out)
        print(f"i={i:>2}: '{ch}' nhìn trước '{peek}' -> token {token:<4} thêm {out!r:<5}"
              f" i -> {i + step:>2} | tạm: {''.join(parts)!r}")
        i += step
    result = "".join(parts)
    print("Kết quả:", repr(result))
    return result

trace_interpret("(al)G(al)()()G")

command = '(al)G(al)()()G', len = 14
i= 0: '(' nhìn trước 'a' -> token (al) thêm 'al'  i ->  4 | tạm: 'al'
i= 4: 'G' nhìn trước '—' -> token G    thêm 'G'   i ->  5 | tạm: 'alG'
i= 5: '(' nhìn trước 'a' -> token (al) thêm 'al'  i ->  9 | tạm: 'alGal'
i= 9: '(' nhìn trước ')' -> token ()   thêm 'o'   i -> 11 | tạm: 'alGalo'
i=11: '(' nhìn trước ')' -> token ()   thêm 'o'   i -> 13 | tạm: 'alGaloo'
i=13: 'G' nhìn trước '—' -> token G    thêm 'G'   i -> 14 | tạm: 'alGalooG'
Kết quả: 'alGalooG'


'alGalooG'

In [12]:
trace_interpret("G()(al)")

command = 'G()(al)', len = 7
i= 0: 'G' nhìn trước '—' -> token G    thêm 'G'   i ->  1 | tạm: 'G'
i= 1: '(' nhìn trước ')' -> token ()   thêm 'o'   i ->  3 | tạm: 'Go'
i= 3: '(' nhìn trước 'a' -> token (al) thêm 'al'  i ->  7 | tạm: 'Goal'
Kết quả: 'Goal'


'Goal'

## 7. Code nộp LeetCode

Bản quét một lượt:

In [13]:
class Solution:
    def interpret(self, command: str) -> str:
        parts = []
        i = 0
        while i < len(command):
            if command[i] == "G":
                parts.append("G")
                i += 1
            elif command[i + 1] == ")":
                parts.append("o")
                i += 2
            else:
                parts.append("al")
                i += 4
        return "".join(parts)

sol = Solution()
assert sol.interpret("G()(al)") == "Goal"
assert sol.interpret("G()()()()(al)") == "Gooooal"
assert sol.interpret("(al)G(al)()()G") == "alGalooG"
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


Bản một dòng bằng `replace` (cũng AC):

In [14]:
class SolutionReplace:
    def interpret(self, command: str) -> str:
        return command.replace("()", "o").replace("(al)", "al")

assert SolutionReplace().interpret("(al)G(al)()()G") == "alGalooG"
print("SolutionReplace PASS")

SolutionReplace PASS


## 8. Các lời giải khác

### 8.1. Bảng tra token bằng `dict` + `startswith`

Tổng quát hóa: đặt bảng `{token: bản_dịch}`. Tại mỗi vị trí, thử từng token xem có bắt đầu tại `i` không. Muốn đổi ngôn ngữ chỉ cần sửa bảng, không sửa vòng lặp.

In [15]:
TOKENS = {"G": "G", "()": "o", "(al)": "al"}

def interpret_table(command: str, table: dict = TOKENS) -> str:
    parts = []
    i = 0
    while i < len(command):
        for token, meaning in table.items():
            if command.startswith(token, i):
                parts.append(meaning)
                i += len(token)
                break
        else:
            raise ValueError(f"token lạ tại vị trí {i}: {command[i:]!r}")
    return "".join(parts)

print(interpret_table("(al)G(al)()()G"))
try:
    interpret_table("G(x)")
except ValueError as e:
    print("ValueError:", e)

alGalooG
ValueError: token lạ tại vị trí 1: '(x)'


Cú pháp **`for ... else`**: khối `else` của vòng `for` chạy khi vòng lặp kết thúc **mà không gặp `break`** — ở đây nghĩa là "không token nào khớp". Đây là cách gọn để báo input sai.

In [16]:
for x in [1, 2, 3]:
    if x == 2:
        print("gặp 2, break")
        break
else:
    print("không in dòng này")

for x in [1, 3]:
    if x == 2:
        break
else:
    print("không gặp break -> chạy else")

gặp 2, break
không gặp break -> chạy else


### 8.2. Biểu thức chính quy `re.sub` với hàm thay thế

`re.sub(pattern, repl, s)` thay mọi chỗ khớp `pattern`. `repl` có thể là **hàm** nhận đối tượng khớp `m` và trả chuỗi thay thế. Mẫu `r"\(\)|\(al\)"` nghĩa là "`()` hoặc `(al)`" (dấu `\(` thoát ký tự ngoặc vì `(` có nghĩa đặc biệt trong regex).

In [17]:
import re

def interpret_regex(command: str) -> str:
    return re.sub(r"\(\)|\(al\)", lambda m: "o" if m.group() == "()" else "al", command)

print(re.findall(r"G|\(\)|\(al\)", "(al)G(al)()()G"), "<- các token")
print(interpret_regex("G()(al)"))
print(interpret_regex("(al)G(al)()()G"))

['(al)', 'G', '(al)', '()', '()', 'G'] <- các token
Goal
alGalooG


### 8.3. Bộ đệm token (buffer) — gom ký tự đến khi khớp

Gom từng ký tự vào `buf`; khi `buf` trùng một token trong bảng thì dịch và xóa `buf`. Không cần nhìn trước, hợp khi các token **không phải tiền tố của nhau** (`"G"`, `"()"`, `"(al)"` thỏa: `"("` chưa phải token).

In [18]:
def interpret_buffer(command: str) -> str:
    parts, buf = [], ""
    for ch in command:
        buf += ch
        if buf in TOKENS:
            parts.append(TOKENS[buf])
            buf = ""
    return "".join(parts)

print(interpret_buffer("G()(al)"))
print(interpret_buffer("(al)G(al)()()G"))

Goal
alGalooG


### 8.4. Mẹo dựa vào ký tự: chỉ xét `)`

Mẹo khéo: duyệt từng ký tự; `G` → thêm `G`; `)` → nếu ký tự **trước** nó là `(` thì thêm `o`, ngược lại thêm `al`; các ký tự khác (`(`, `a`, `l`) bỏ qua.

In [19]:
def interpret_close_paren(command: str) -> str:
    parts = []
    for i, ch in enumerate(command):
        if ch == "G":
            parts.append("G")
        elif ch == ")":
            parts.append("o" if command[i - 1] == "(" else "al")
    return "".join(parts)

print(interpret_close_paren("G()()()()(al)"))
print(interpret_close_paren("(al)G(al)()()G"))

Gooooal
alGalooG


### 8.5. Bảng so sánh các cách

Gọi `n = len(command)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Nhận xét |
|---|---|---|---|---|
| 2 lần `replace` | thay chuỗi con | `O(n)` (2 lượt) | `O(n)` | ngắn nhất; an toàn vì token không chồng lấn |
| **Quét + lookahead** | nhìn trước 1 ký tự | `O(n)` (1 lượt) | `O(n)` | **rõ ràng, nên học** |
| Bảng `dict` + `startswith` | thử từng token | `O(n · T)` | `O(n)` | dễ mở rộng (T = số token) |
| `re.sub` | regex | `O(n)` | `O(n)` | gọn khi quen regex |
| Bộ đệm token | gom tới khi khớp | `O(n)` | `O(n)` | cần token không là tiền tố của nhau |
| Chỉ xét `)` | mẹo vị trí | `O(n)` | `O(n)` | khéo nhưng phụ thuộc bảng cụ thể |

(Bộ nhớ `O(n)` là cho chuỗi kết quả — bắt buộc phải tạo.)

## 9. Vì sao lời giải quét đúng?

**Invariant:** ở đầu mỗi vòng `while`, `i` là vị trí **bắt đầu của một token**, và `"".join(parts)` là bản dịch của `command[:i]`.

- Ban đầu `i = 0` là đầu token đầu tiên (input hợp lệ), `parts` rỗng ↔ `command[:0]` rỗng.
- Tại `i`, ba token bắt đầu bằng hai loại ký tự: `G` (chỉ token `G`) và `(` (token `()` hoặc `(al)`). Hai token ngoặc **khác nhau ở ký tự thứ hai** (`)` vs `a`), nên nhìn `command[i+1]` xác định duy nhất token.
- Ta thêm đúng bản dịch và nhảy đúng độ dài token → `i` mới lại là đầu token kế tiếp, invariant giữ.

Khi `i == len(command)`, `parts` là bản dịch của toàn chuỗi.

**Vì sao 2 lần `replace` đúng:** `"()"` và `"(al)"` không chồng lấn và không chứa nhau; kết quả thay (`"o"`, `"al"`) không chứa `(` hay `)` nên không tạo token mới. Mỗi token ngoặc bị thay đúng một lần, `G` không bị đụng tới.

## 10. Độ phức tạp

Gọi `n = len(command)`.

- **Quét một lượt:** mỗi ký tự được đọc `O(1)` lần (nhảy qua token) → **Time `O(n)`**. `"".join` cũng `O(n)`. **Space `O(n)`** cho `parts` và chuỗi kết quả.
- **Hai lần `replace`:** mỗi lần `O(n)` → tổng `O(n)`, tạo hai chuỗi trung gian.
- **Bảng `dict`:** mỗi vị trí thử tối đa `T` token, mỗi lần `startswith` tối đa độ dài token → `O(n · T · L)` với `L` = độ dài token dài nhất; ở bài này `T = 3`, `L = 4` nên vẫn là `O(n)`.

## 11. Bộ test

1. 3 ví dụ của đề.
2. Biên: chỉ một token (`"G"`, `"()"`, `"(al)"`).
3. Toàn một loại token lặp lại.
4. Độ dài tối đa ~100.
5. Đối chiếu 6 cách trên chuỗi sinh ngẫu nhiên từ các token (luôn hợp lệ), so với "bản dịch chuẩn" tính từ list token.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS -> {actual!r}")

f = interpret_scan
check("đề 1", f("G()(al)"), "Goal")
check("đề 2", f("G()()()()(al)"), "Gooooal")
check("đề 3", f("(al)G(al)()()G"), "alGalooG")
check("chỉ G", f("G"), "G")
check("chỉ ()", f("()"), "o")
check("chỉ (al)", f("(al)"), "al")
check("toàn G", f("GGGG"), "GGGG")
check("toàn (al)", f("(al)(al)"), "alal")
check("dài 100", f("()" * 50), "o" * 50)

đề 1                   PASS -> 'Goal'
đề 2                   PASS -> 'Gooooal'
đề 3                   PASS -> 'alGalooG'
chỉ G                  PASS -> 'G'
chỉ ()                 PASS -> 'o'
chỉ (al)               PASS -> 'al'
toàn G                 PASS -> 'GGGG'
toàn (al)              PASS -> 'alal'
dài 100                PASS -> 'oooooooooooooooooooooooooooooooooooooooooooooooooo'


In [21]:
import random

rng = random.Random(1678)
methods = [interpret_replace, interpret_table, interpret_regex, interpret_buffer,
           interpret_close_paren, Solution().interpret, SolutionReplace().interpret]
pieces = list(TOKENS.items())
for _ in range(500):
    chosen = [rng.choice(pieces) for _ in range(rng.randint(1, 25))]
    command = "".join(tok for tok, _ in chosen)
    expected = "".join(meaning for _, meaning in chosen)
    assert interpret_scan(command) == expected
    for meth in methods:
        assert meth(command) == expected, (getattr(meth, "__name__", meth), command)
print("8 cách khớp bản dịch chuẩn trên 500 chuỗi ngẫu nhiên hợp lệ")

8 cách khớp bản dịch chuẩn trên 500 chuỗi ngẫu nhiên hợp lệ


Kỹ thuật test ở trên gọi là **sinh từ cấu trúc**: thay vì tạo chuỗi ngẫu nhiên (có thể không hợp lệ), ta chọn ngẫu nhiên **danh sách token**, rồi tạo cả input lẫn đáp án mong đợi từ cùng danh sách đó.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Thay từng ký tự `(` → `o`, bỏ `)`

Ý tưởng "mỗi `()` là `o`" làm nhiều người thay `(` thành `o` và xóa `)`. Nhưng `(al)` sẽ thành `oal`.

In [22]:
def wrong_char_replace(command):
    return command.replace("(", "o").replace(")", "")

print("sai :", wrong_char_replace("G()(al)"))
print("đúng:", interpret_scan("G()(al)"))

sai : Gooal
đúng: Goal


### Lỗi 2: Thứ tự `replace` sai khi bảng có chồng lấn

Ở bài gốc thứ tự nào cũng đúng. Nhưng giả sử thêm token `"(al)()"` → `"X"`: nếu thay `"()"` trước thì `"(al)()"` bị phá thành `"(al)o"` và không bao giờ khớp `X`. Bài học: khi token chồng lấn, **thay token dài trước**, hoặc dùng bộ quét.

In [23]:
s = "(al)()"
wrong = s.replace("()", "o").replace("(al)()", "X").replace("(al)", "al")
right = s.replace("(al)()", "X").replace("()", "o").replace("(al)", "al")
print("thay () trước     :", wrong)
print("thay token dài trước:", right)

thay () trước     : alo
thay token dài trước: X


### Lỗi 3: Dùng `for i in range(len(command))` rồi cố nhảy `i`

Gán `i += 4` trong `for` không có tác dụng với vòng kế tiếp; các ký tự `a`, `l`, `)` sẽ bị xử lý lại như token mới.

In [24]:
def wrong_for_jump(command):
    parts = []
    for i in range(len(command)):
        if command[i] == "G":
            parts.append("G")
        elif command[i] == "(":
            if command[i + 1] == ")":
                parts.append("o")
                i += 2          # vô tác dụng
            else:
                parts.append("al")
                i += 4          # vô tác dụng
    return "".join(parts)

print("tình cờ đúng:", wrong_for_jump("G()(al)"), "<- chỉ đúng vì ký tự bên trong token bị bỏ qua")
print("=> code này dựa vào may mắn; với bảng token khác (ví dụ token 'GG') sẽ sai. Dùng while.")

tình cờ đúng: Goal <- chỉ đúng vì ký tự bên trong token bị bỏ qua
=> code này dựa vào may mắn; với bảng token khác (ví dụ token 'GG') sẽ sai. Dùng while.


Ở đây code "tình cờ" ra đúng vì các ký tự bên trong token (`)`, `a`, `l`) không khớp nhánh nào nên bị bỏ qua — nhưng ý định "nhảy `i`" là sai và người đọc sẽ hiểu nhầm. Đó là kiểu lỗi nguy hiểm nhất: **đúng mà không biết vì sao**.

### Lỗi 4: Cộng chuỗi trong vòng lặp với dữ liệu lớn

`res += "o"` đúng logic nhưng mỗi lần có thể tạo chuỗi mới. Với `n <= 100` không sao; thói quen tốt là `list.append` + `"".join`.

### Lỗi 5: Quên `G` khi dùng mẹo "chỉ xét `)`"

In [25]:
def wrong_forget_g(command):
    parts = []
    for i, ch in enumerate(command):
        if ch == ")":
            parts.append("o" if command[i - 1] == "(" else "al")
    return "".join(parts)

print("sai :", wrong_forget_g("G()(al)"))
print("đúng:", interpret_close_paren("G()(al)"))

sai : oal
đúng: Goal


## 13. Mẹo nhận dạng pattern

- "Chuỗi gồm các mảnh từ một bảng cố định, dịch từng mảnh" → **lexer**: quét trái sang phải, nhận diện token, nhảy qua.
- Các token phân biệt được bằng **k ký tự đầu** → nhìn trước `k - 1` ký tự (ở đây `k = 2`).
- Bước nhảy thay đổi → **`while` + tự tăng `i`**.
- Bảng token không chồng lấn và kết quả thay không tạo token mới → `replace` là đủ.
- Xây chuỗi kết quả → `list` + `"".join`.

Bài liên quan: 1108 Defanging an IP Address, 1309 Decrypt String from Alphabet to Integer Mapping, 13 Roman to Integer (nhìn trước 1 ký tự), 20 Valid Parentheses, 394 Decode String.

## 14. Biến thể

### Biến thể 1: Mã hóa ngược (Goal → command)

Từ chuỗi gồm `G`, `o`, `al`, tạo lại `command`. Cũng là quét với lookahead: `a` luôn đi với `l`.

In [26]:
def encode_goal(text: str) -> str:
    parts, i = [], 0
    while i < len(text):
        if text[i] == "G":
            parts.append("G"); i += 1
        elif text[i] == "o":
            parts.append("()"); i += 1
        else:                      # "al"
            parts.append("(al)"); i += 2
    return "".join(parts)

print(encode_goal("alGalooG"))
print(interpret_scan(encode_goal("Gooooal")) == "Gooooal")

(al)G(al)()()G
True


### Biến thể 2: Đếm từng loại token

In [27]:
from collections import Counter

def count_tokens(command: str) -> Counter:
    return Counter(re.findall(r"G|\(\)|\(al\)", command))

print(count_tokens("(al)G(al)()()G"))

Counter({'(al)': 2, 'G': 2, '()': 2})


### Biến thể 3: Bảng token tùy ý, kiểm tra hợp lệ

Dùng lại `interpret_table` với bảng khác — ví dụ một "ngôn ngữ" mini.

In [28]:
mini = {"<3": "love", ":)": "happy", " ": " "}
print(interpret_table("<3 :)", mini))

love happy


## 15. Bài tập tự luyện

Các cell TODO chỉ định nghĩa hàm, không được gọi khi chạy notebook.

### Bài 1 — Trace bằng tay

`command = "()(al)G()"`. Lập bảng `i, ký tự, nhìn trước, token, i mới` và cho kết quả.

### Bài 2 — Viết bản `replace`

In [29]:
def exercise_replace(command: str) -> str:
    # TODO: hai lần replace
    pass

# assert exercise_replace("G()(al)") == "Goal"

### Bài 3 — Viết lại bộ quét lookahead

In [30]:
def exercise_scan(command: str) -> str:
    # TODO 1: parts = [], i = 0
    # TODO 2: while: G -> +1; '(' + ')' -> +2; còn lại -> +4
    # TODO 3: return "".join(parts)
    pass

# assert exercise_scan("(al)G(al)()()G") == "alGalooG"

### Bài 4 — Defanging an IP Address (LeetCode 1108)

Thay mỗi `"."` bằng `"[.]"`. Viết bằng **vòng lặp** (không dùng `replace`).

In [31]:
def exercise_defang(address: str) -> str:
    # TODO: duyệt từng ký tự, append "[.]" hoặc ký tự gốc
    pass

# assert exercise_defang("1.1.1.1") == "1[.]1[.]1[.]1"

### Bài 5 — Biến thể: tách command thành list token

In [32]:
def exercise_tokenize(command: str) -> list[str]:
    # TODO: giống bộ quét nhưng append chính token, không phải bản dịch
    pass

# assert exercise_tokenize("G()(al)") == ["G", "()", "(al)"]

### Bài 6 — Mini challenge: Decrypt String from Alphabet to Integer Mapping (LeetCode 1309)

`"1"`..`"9"` → `"a"`..`"i"`, `"10#"`..`"26#"` → `"j"`..`"z"`. Ví dụ `"10#11#12"` → `"jkab"`. Gợi ý: nhìn trước **2** ký tự xem có `#` không.

In [33]:
def exercise_freq_alphabets(s: str) -> str:
    # TODO: while i < len(s): nếu i + 2 < len(s) và s[i+2] == '#': lấy 2 chữ số, nhảy 3; ngược lại 1 chữ số, nhảy 1
    pass

# assert exercise_freq_alphabets("10#11#12") == "jkab"
# assert exercise_freq_alphabets("1326#") == "acz"

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: bắt đầu `i = 0`, ký tự `(`, nhìn trước `)`.
- Mức 2: `i: 0 → 2 → 6 → 7 → 9`.
- Mức 3: token `()`, `(al)`, `G`, `()` → `"oalGo"`.

**Bài 2**
- Mức 1: method chaining.
- Mức 2: `"()"` → `"o"`.
- Mức 3: `.replace("(al)", "al")`.

**Bài 3**
- Mức 1: dùng `while`.
- Mức 2: nhánh `elif command[i + 1] == ")"`.
- Mức 3: ba nhánh với bước 1, 2, 4.

**Bài 4**
- Mức 1: `parts = []`.
- Mức 2: `parts.append("[.]" if ch == "." else ch)`.
- Mức 3: `"".join(parts)`.

**Bài 5**
- Mức 1: token `G`, `()`, `(al)` đúng là các slice `command[i:i+step]`.
- Mức 2: xác định `step` như bộ quét.
- Mức 3: `tokens.append(command[i:i + step]); i += step`.

**Bài 6**
- Mức 1: đọc trái sang phải, nhìn `s[i + 2]`.
- Mức 2: số `k` → chữ `chr(ord('a') + k - 1)`.
- Mức 3: `int(s[i:i+2])` khi có `#`, `int(s[i])` khi không.

## 17. Đáp án tham khảo

**Bài 1:** `i=0 '(' nhìn ')' → "()" → o, i=2`; `i=2 '(' nhìn 'a' → "(al)" → al, i=6`; `i=6 'G' → G, i=7`; `i=7 '(' nhìn ')' → o, i=9`. Kết quả `"oalGo"`.

In [34]:
def answer_replace(command):
    return command.replace("()", "o").replace("(al)", "al")

def answer_scan(command):
    parts, i = [], 0
    while i < len(command):
        if command[i] == "G":
            parts.append("G"); i += 1
        elif command[i + 1] == ")":
            parts.append("o"); i += 2
        else:
            parts.append("al"); i += 4
    return "".join(parts)

def answer_defang(address):
    parts = []
    for ch in address:
        parts.append("[.]" if ch == "." else ch)
    return "".join(parts)

def answer_tokenize(command):
    tokens, i = [], 0
    while i < len(command):
        if command[i] == "G":
            step = 1
        elif command[i + 1] == ")":
            step = 2
        else:
            step = 4
        tokens.append(command[i:i + step])
        i += step
    return tokens

def answer_freq_alphabets(s):
    parts, i = [], 0
    while i < len(s):
        if i + 2 < len(s) and s[i + 2] == "#":
            k = int(s[i:i + 2]); i += 3
        else:
            k = int(s[i]); i += 1
        parts.append(chr(ord("a") + k - 1))
    return "".join(parts)

assert answer_scan("()(al)G()") == "oalGo"                       # Bài 1
assert answer_replace("G()(al)") == "Goal"
assert answer_scan("(al)G(al)()()G") == "alGalooG"
assert answer_defang("1.1.1.1") == "1[.]1[.]1[.]1"
assert answer_tokenize("G()(al)") == ["G", "()", "(al)"]
assert answer_freq_alphabets("10#11#12") == "jkab"
assert answer_freq_alphabets("1326#") == "acz"
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def interpret(self, command: str) -> str:
        parts, i = [], 0
        while i < len(command):
            if command[i] == "G":
                parts.append("G"); i += 1          # token G
            elif command[i + 1] == ")":
                parts.append("o"); i += 2          # token ()
            else:
                parts.append("al"); i += 4         # token (al)
        return "".join(parts)
```

Bản nhanh: `return command.replace("()", "o").replace("(al)", "al")`.

Câu thần chú: **"Đọc ký tự đầu, nhìn trước một ký tự, biết token, nhảy qua token."**

Checklist trước khi submit:

- [ ] Phân biệt `()` và `(al)` bằng ký tự **thứ hai**, không thay từng ký tự `(`?
- [ ] Dùng `while` khi bước nhảy thay đổi?
- [ ] Không quên token `G`?
- [ ] Nếu dùng `replace`: hiểu vì sao thứ tự không gây lỗi ở bài này?
- [ ] Xây kết quả bằng `list` + `"".join`?